# Assignment Coverage Checklist ✅

This notebook implements all required parts of the HW3 assignment. Below is a mapping from assignment items to notebook sections and saved outputs (files are under `images/`):

1. Question 1 (EBM):
- Data prep & loaders: See cell <Data loading and a quick peek>. Saved sample: `images/mnist_sample.png`.
- Model architecture: `ebm_model.ConvEnergyModel` implements Table 1.
- Langevin sampling: `ebm_sampling.LangevinSampler` and `sample_from_noise`.
- Training: `ebm_train.train` and `ebm_train.train_interactive`. Checkpoints: `images/ebm/ebm_ckpt.pt`.
- Post-training sampling: `ebm_infer.generate_and_denoise` saves `ebm_samples_final.png` and denoising outputs for each noise level `ebm_denoised_<nl>.png`.
- Denoising experiments for noise levels 0.2, 0.4, 0.6: use the dedicated cell named "EBM: Denoising at multiple noise levels".

2. Question 2 (NCSN):
- Data prep & normalization to [-1,1]: Done in `full_train_ncsn`.
- ScoreNet architecture: `ncsn_model.ScoreNet` (U-Net with AdaptiveResBlock and FiLM).
- DSM loss: `ncsn_loss.dsm_loss` (weighted DSM).
- ALD sampling: `ncsn_sampling.annealed_langevin_dynamics` and `sample`.
- Conditional model: `ScoreNet` supports `cfg.conditional=True` with `nn.Embedding` for labels; training via `full_train_ncsn(..., conditional=True)`.
- Conditional generation grid: use the cell "NCSN Conditional: Generate a 10x10 class-conditional grid" which saves `images/ncsn_cond_grid/ncsn_cond_grid.png`.
- Denoising at noise levels 0.2/0.4/0.6: use `ncsn_infer.ncsn_generate_and_denoise` (cell in notebook calls this).
- Trajectory visualization: `ncsn_traj_demo` frames and GIF generation cell.

3. Theory: Comprehensive markdown sections for all Q1 and Q2 subquestions with derivations and practical tips.

If you want, I can:
- Add more analysis text for each saved image to include in the report, or
- Prepare a final 'Results' folder with a README summarizing which images to include in the report.

Next step: tell me which of these optional follow-ups you want me to do next.

# Theory Questions: Energy-Based Models (EBM)

**Q1.1:** To generate a face of a young man using an EBM trained on faces, we can use conditional sampling or guide the sampling process by conditioning on attributes (e.g., age, gender) if available. If not, we can use attribute classifiers to filter or steer the generated samples.

**Q1.2:** Rejection Sampling is a method to sample from a target distribution $p(x)$ using a proposal distribution $q(x)$ and a constant $M$ such that $p(x) \leq M q(x)$ for all $x$. We sample $x \sim q(x)$ and accept with probability $p(x)/(M q(x))$. If $M=2$, the acceptance rate is $1/2$ (i.e., $1/M$).

**Q1.3:**
a) The training process pushes the energy $E_\theta(x)$ down for real data and up for generated (fake) samples.
b) The expectation over the model (second term) is intractable due to the partition function. Contrastive Divergence (CD) approximates this by running a short Markov chain (e.g., Langevin) starting from data, making training feasible.

# Detailed Theory: Energy-Based Models (EBM)

## Q1.1 — Generating a face with a specific attribute (e.g., a young man)

Short answer: use a conditional EBM or steer the sampling via attribute conditioning (classifier guidance or conditioning variables).

Explanation: If the training set contains attribute labels (age, gender, etc.), you can train an EBM conditioned on the attribute y, i.e., model p_θ(x | y) ∝ exp(-E_θ(x, y)). During sampling (e.g., Langevin dynamics) you fix y to the desired attribute and sample x that minimizes the conditional energy. If labels are not available, you can use a pretrained attribute classifier C(x) and either: (a) run classifier-guided Langevin (adjust gradients to increase classifier score for the desired attribute), or (b) reject samples that don’t match the attribute. Both are practical approaches used in conditional generation and guided sampling.

## Q1.2 — Rejection Sampling (explanation + acceptance rate)

Rejection sampling algorithm (brief):
1. Choose a proposal distribution q(x) and a constant M such that p(x) ≤ M q(x) for all x.
2. Repeat: draw x ∼ q(x), draw u ∼ Uniform(0, 1). Accept x if u ≤ p(x) / (M q(x)).

Acceptance probability: E_q[ p(x) / (M q(x)) ] = (1/M) ∫ p(x) dx = 1/M (for normalized p). So the expected acceptance rate equals 1/M. If M = 2, the acceptance rate is 1/2 (50%).

## Q1.3 — Gradient of the log-likelihood and training behavior

Recall (sketch) that for an unnormalized energy model p_θ(x) = exp(-E_θ(x)) / Z_θ, the gradient of the log-likelihood (for a single data point x) can be written as:

∇_θ log p_θ(x) = -∇_θ E_θ(x) + E_{x'∼p_θ}[∇_θ E_θ(x')].

a) What training tries to do:
- The first term (−∇_θ E_θ(x)) pushes parameters so that the energy of data x is decreased (makes data more likely).
- The second term (the model expectation) pushes parameters so that the energy assigned to typical model-generated samples is increased (reduces probability mass in regions where the model currently places mass incorrectly).

Net effect: energies at data points are lowered while energies at model-generated points are raised — this separates data and model distributions.

b) Practical challenge and Contrastive Divergence (CD):
- The expectation E_{x'∼p_θ}[·] requires sampling from p_θ, which is intractable because p_θ depends on the unknown normalization constant Z_θ and may require long MCMC chains to mix.
- Contrastive Divergence approximates this expectation by initializing a short MCMC chain from a data point (or small number of steps from data) and using the resulting sample as an approximate draw from the model. CD is biased but computationally efficient and often works well in practice for training EBMs and related undirected models.

# Complete Theory (EBM) — Detailed Derivations, Algorithms, and Practical Tips

## 1. Conditional generation with EBMs (Q1.1 expanded)

If you want to sample a face with a specific attribute (e.g., a young man), the cleanest approach is to train or fine-tune a conditional EBM p_θ(x | y) ∝ exp(-E_θ(x, y)). During sampling you fix y (the attribute label) and run Langevin dynamics on x to find low-energy images for that conditioning. Practical alternatives when labels are unavailable:
- Classifier-guided sampling: train a separate attribute classifier C(x) and add its gradient to the Langevin update (steer samples toward high classifier score). This is often used in guided diffusion and classifier-guided sampling.
- Rejection or filtering: generate unconditional samples and reject those that do not satisfy the attribute (inefficient for rare attributes).

## 2. Rejection sampling details and acceptance rate (Q1.2 expanded)

Given target density p(x) (normalized) and a proposal q(x) and constant M with p(x) ≤ M q(x), the algorithm samples x ~ q, u ~ Uniform(0,1) and accepts if u ≤ p(x)/(M q(x)). The acceptance probability (averaged over q) is:

\mathbb{E}_q[ p(x) / (M q(x)) ] = \frac{1}{M} \int p(x) dx = 1 / M,

so if M = 2 the acceptance rate is 50%. Note: For unnormalized EBM densities p̃(x) = exp(-E(x)), the same approach requires computing a normalized p (or an M bounding constant using a normalized surrogate). Rejection sampling is inefficient in high-dimensions because a tight M is hard to find.

## 3. Maximum likelihood gradient and Contrastive Divergence (Q1.3 expanded)

Start with p_θ(x) = exp(-E_θ(x)) / Z_θ. The gradient of log-likelihood for a datapoint x is:

\nabla_θ \log p_θ(x) = -\nabla_θ E_θ(x) + \mathbb{E}_{x'\sim p_θ}[\nabla_θ E_θ(x')].

Interpretation:
- The first term pushes energy at real data downward (makes data more probable).
- The second term (model expectation) raises energy in regions where the model currently places mass. This term is expensive because sampling from p_θ is intractable.

Contrastive Divergence (CD-k): approximate the model expectation with samples obtained by running k steps of MCMC starting from the data point. Basic CD-k pseudocode:
```
for batch x_real in data:
    x_neg = x_real.clone()
    for i in range(k):
        x_neg = MCMC_step(x_neg)  # e.g., Langevin update
    # Use x_neg as approximate sample from model in gradient estimation
    update θ using −∇_θ E(x_real) + ∇_θ E(x_neg)
```
CD is biased but efficient; larger k reduces bias at higher compute cost.

## 4. Langevin dynamics (practical notes & pseudocode)

Langevin sampling for images implements a discretized overdamped Langevin SDE: dx = −α ∇_x E_θ(x) dt + sqrt(2α) dW. In practice use step-size η and add Gaussian noise:

```python
# x: init tensor in [0,1] or [-1,1] depending on normalization
for t in range(T):
    x.requires_grad_(True)
    e = model(x).sum()  # scalar energy
    grad = torch.autograd.grad(e, x)[0]
    x = x - 0.5 * eta * grad + torch.randn_like(x) * sqrt(eta)
    x = clamp(x, min_val, max_val)  # keep valid image range
```
Important practical tips:
- Use small η (e.g., 1e-3–1e-1) and many steps T for high quality; for speed use fewer steps but expect lower quality.
- Add clamping to valid pixel ranges after each step (or use reflective boundary), but clamping changes dynamics — consider projecting to allowed domain carefully.
- Use more steps when starting from pure noise; fewer when denoising noisy images.
- Use a schedule (decrease η) or add a temperature parameter if exploration is insufficient.

## 5. Diagnostics and best practices

- Monitor E_real and E_fake (mean energies) during training — they should diverge in the right direction (E_real ↓ and E_fake ↑), but watch for trivial solutions where both become small/large (use the reg term).
- Save sample grids and denoising examples after each epoch to inspect quality progression.
- If training collapses or samples are poor: reduce learning rate, check Langevin step size and number of steps, ensure correct gradient computation on inputs (no in-place ops), and check regularization λ.

## 6. Implementation notes for MNIST (quick checklist)

- Normalize data to [0,1] for EBM (or if using other conventions, be consistent between training and sampling).
- Use batch size 64–128, epochs 10–50 depending on compute.
- Langevin: T = 60–200 steps (training and sampling), η tuned per experiment, try adding gradient clipping if unstable.
- Save checkpoints and sample images every epoch so the heavy-run notebook can pick up outputs.

# CA3 HW3: Energy-Based and Score-Based Models on MNIST

**Objectives**

- Implement EBM with Langevin sampling and contrastive divergence.
- Implement NCSN with weighted DSM and annealed Langevin dynamics (unconditional + conditional).
- Provide training, sampling, and denoising pipelines with reproducibility hooks.

**Structure**

1. Setup and configuration
2. Data loading and visualization
3. EBM model, training, sampling, denoising
4. NCSN model, training, sampling (ALD), denoising, conditional variant
5. Results logging placeholders
6. Reproducibility notes


In [ ]:
# Setup and Configuration
import os, random
import numpy as np
import torch
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_ROOT = PROJECT_ROOT / "data" / "mnist"


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# Enable relative imports in notebook
import sys
sys.path.insert(0, str(Path.cwd()))

In [ ]:
# Install requirements (run this in Colab)
!pip install -r ../requirements.txt

In [ ]:
# Configs
from config import DataConfig, EBMConfig, NCSNConfig, RunPaths

data_cfg = DataConfig()
ebm_cfg = EBMConfig(device=device)
ncsn_cfg = NCSNConfig(device=device)
paths = RunPaths()
paths.ensure()
data_cfg, ebm_cfg, ncsn_cfg

In [ ]:
# Data loading and a quick peek
from data import mnist_dataloaders
from torchvision.utils import make_grid, save_image
import matplotlib.pyplot as plt

train_loader, test_loader = mnist_dataloaders(data_cfg, normalize_to_minus1_1=False)
images, labels = next(iter(train_loader))
grid = make_grid(images[:16], nrow=4)
save_image(grid.detach().cpu(), paths.images / "mnist_sample.png")
plt.figure(figsize=(4, 4))
plt.axis("off")
img_arr = grid.permute(1, 2, 0).detach().cpu().numpy()
plt.imshow(img_arr)
plt.show()

In [ ]:
# EBM model, sampler, and a short training utility (configurable)
# Prefer using script helper `train_interactive` from `ebm_train` for reuse
from ebm_model import ConvEnergyModel
from ebm_sampling import LangevinSampler, sample_from_noise
from torch import optim
from tqdm import tqdm
from torchvision.utils import save_image
from ebm_train import train_interactive as train_ebm

# Note: `train_ebm` now refers to `train_interactive` in `ebm_train.py` which
# performs a short interactive run, saves sample grids to `paths.images` and
# returns (model, history). Use `full_train_ebm` from `ebm_train` for full runs.

In [ ]:
# Full EBM Training Pipeline
from dataclasses import asdict
from pathlib import Path
from typing import Dict, Any
from torch import optim
from tqdm import tqdm
from config import DataConfig, EBMConfig, RunPaths
from data import mnist_dataloaders
from ebm_model import ConvEnergyModel
from ebm_sampling import LangevinSampler, sample_from_noise
from utils import save_grid, set_seed, ensure_dir, write_run_info


def full_train_ebm(cfg_data: DataConfig, cfg_model: EBMConfig, output_dir: Path) -> Dict[str, Any]:
    set_seed(cfg_data.seed)
    train_loader, test_loader = mnist_dataloaders(cfg_data)
    device = cfg_model.device

    model = ConvEnergyModel().to(device)
    optimizer = optim.Adam(model.parameters(), lr=cfg_model.lr)
    sampler = LangevinSampler(model, cfg_model)

    history = {"loss": [], "E_real": [], "E_fake": []}
    ensure_dir(output_dir)
    checkpoint_path = output_dir / "ebm_ckpt.pt"
    write_run_info(
        output_dir,
        configs={"data": asdict(cfg_data), "model": asdict(cfg_model)},
        notes={"script": "notebook full_train_ebm"},
        device=str(device),
    )

    for epoch in range(1, cfg_model.epochs + 1):
        progress = tqdm(
            train_loader, desc=f"EBM Epoch {epoch}/{cfg_model.epochs}", leave=False
        )
        for step, (x_real, _) in enumerate(progress, start=1):
            x_real = x_real.to(device)
            x_fake = sampler(torch.rand_like(x_real))

            E_real = model(x_real)
            E_fake = model(x_fake)

            data_term = E_real.mean() - E_fake.detach().mean()
            reg_term = cfg_model.lambda_reg * (
                E_real.pow(2).mean() + E_fake.detach().pow(2).mean()
            )
            loss = data_term + reg_term

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            history["loss"].append(loss.item())
            history["E_real"].append(E_real.mean().item())
            history["E_fake"].append(E_fake.mean().item())

            if step % cfg_model.log_interval == 0:
                progress.set_postfix(loss=f"{loss.item():.3f}")

        # Save training samples each epoch
        # Sampling requires gradients to compute input gradients via autograd
        samples = sample_from_noise(
            model, cfg_model, (cfg_model.sample_grid, 1, 28, 28)
        )
        save_grid(samples.detach().cpu(), output_dir / f"ebm_samples_epoch{epoch}.png", nrow=4)

        # Denoising a few test digits via Langevin starting from noisy images
        x_test, _ = next(iter(test_loader))
        x_test = x_test[: cfg_model.sample_grid].to(device)
        noise = torch.randn_like(x_test) * 0.3
        noisy = (x_test + noise).clamp(0.0, 1.0)
        denoised = sampler(noisy)
        save_grid(x_test.detach().cpu(), output_dir / f"ebm_real_epoch{epoch}.png", nrow=4)
        save_grid(noisy.detach().cpu(), output_dir / f"ebm_noisy_epoch{epoch}.png", nrow=4)
        save_grid(denoised.detach().cpu(), output_dir / f"ebm_denoised_epoch{epoch}.png", nrow=4)

        torch.save(
            {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "epoch": epoch,
            },
            checkpoint_path,
        )

    # Save simple visualizations of loss and energies
    try:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(6, 4))
        plt.plot(history["loss"], label="loss")
        plt.xlabel("Step")
        plt.ylabel("Loss")
        plt.title("EBM Loss")
        plt.legend()
        plt.tight_layout()
        plt.savefig(output_dir / "ebm_loss.png")
        plt.close()

        plt.figure(figsize=(6, 4))
        plt.plot(history["E_real"], label="E_real")
        plt.plot(history["E_fake"], label="E_fake")
        plt.xlabel("Step")
        plt.ylabel("Energy")
        plt.title("EBM Energies")
        plt.legend()
        plt.tight_layout()
        plt.savefig(output_dir / "ebm_energy.png")
        plt.close()
    except Exception:
        # Best-effort visualization; continue even if matplotlib is unavailable.
        pass

    return history


# Example: full_ebm_cfg = EBMConfig(epochs=10)
# full_train_ebm(DataConfig(), full_ebm_cfg, paths.images / "ebm")

# EBM: Denoising at multiple noise levels

In [ ]:
# Denoising demonstration
from ebm_infer import generate_and_denoise
# Assuming model is trained and checkpoint saved
ebm_ckpt = paths.images / "ebm" / "ebm_ckpt.pt"
if ebm_ckpt.exists():
    generate_and_denoise(ebm_ckpt, paths.images / "ebm_denoise", DataConfig(), EBMConfig())
    print("Denoising images saved to", paths.images / "ebm_denoise")
else:
    print("EBM checkpoint not found. Run full training first.")

# EBM: Langevin Sampling Trajectory Visualization

In [ ]:
# Trajectory visualization
from ebm_infer import sample_and_save_trajectory
ebm_ckpt = paths.images / "ebm" / "ebm_ckpt.pt"
if ebm_ckpt.exists():
    traj_dir = paths.images / "ebm_trajectory"
    sample_and_save_trajectory(ebm_ckpt, traj_dir, EBMConfig(), record_every=5)
    print("Trajectory frames saved to", traj_dir)
    # Optionally create GIF
    make_gif_from_frames(traj_dir, paths.images / "ebm_trajectory.gif", fps=6)
else:
    print("EBM checkpoint not found.")

In [ ]:
# NCSN model, DSM loss, and ALD sampling utilities
# Prefer using script helper `train_interactive` from `ncsn_train` for reuse
from ncsn_model import ScoreNet
from ncsn_loss import dsm_loss
from ncsn_sampling import sample as ncsn_sample
from torchvision.utils import save_image
from ncsn_train import train_interactive as train_ncsn

# Note: `train_ncsn` now refers to `train_interactive` in `ncsn_train.py` which
# performs a short interactive run, saves sample grids to `paths.images` and
# returns (model, history). Use `full_train_ncsn` from `ncsn_train` for full runs.

In [ ]:
# Full NCSN Training Pipeline
import matplotlib.pyplot as plt
from dataclasses import asdict
from pathlib import Path
from typing import Dict, Any, Optional

import torch
from torch import optim
from tqdm import tqdm

from config import NCSNConfig, DataConfig, RunPaths
from data import mnist_dataloaders
from ncsn_model import ScoreNet
from ncsn_loss import dsm_loss
from ncsn_sampling import sample
from utils import save_grid, set_seed, ensure_dir, write_run_info


def full_train_ncsn(cfg: NCSNConfig, output_dir: Path, conditional: bool = False) -> Dict[str, Any]:
    cfg.conditional = conditional
    set_seed(42)

    data_cfg = DataConfig(
        batch_size=cfg.batch_size, num_workers=cfg.num_workers, channels=cfg.channels
    )
    train_loader, _ = mnist_dataloaders(data_cfg, normalize_to_minus1_1=True)

    device = cfg.device
    model = ScoreNet(cfg).to(device)
    optimizer = optim.Adam(model.parameters(), lr=cfg.lr)
    sigmas = cfg.sigmas

    ensure_dir(output_dir)
    history = {"loss": []}
    checkpoint_path = output_dir / ("ncsn_cond.pt" if conditional else "ncsn.pt")
    write_run_info(
        output_dir,
        configs={
            "data": asdict(data_cfg),
            "model": asdict(cfg),
            "conditional": {"enabled": conditional},
        },
        notes={"script": "notebook full_train_ncsn"},
        device=str(device),
    )

    for epoch in range(1, cfg.epochs + 1):
        progress = tqdm(
            train_loader, desc=f"NCSN Epoch {epoch}/{cfg.epochs}", leave=False
        )
        for x, labels in progress:
            x = x.to(device)
            x = x * 2 - 1  # map to [-1, 1]
            y = labels.to(device) if conditional else None

            loss = dsm_loss(model, x, cfg, sigmas, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            history["loss"].append(loss.item())
            progress.set_postfix(loss=f"{loss.item():.3f}")

        with torch.no_grad():
            y_samples: Optional[torch.Tensor] = None
            if conditional:
                y_samples = torch.arange(0, 16, device=device) % cfg.num_classes
            samples = sample(model, cfg, num_samples=16, y=y_samples)
            samples = (samples + 1) / 2.0
            save_grid(samples, output_dir / f"samples_epoch{epoch}.png", nrow=4)

        torch.save(
            {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "epoch": epoch,
            },
            checkpoint_path,
        )

    # Save loss visualization
    try:
        plt.figure(figsize=(6, 4))
        plt.plot(history["loss"], label="DSM loss")
        plt.xlabel("Step")
        plt.ylabel("Loss")
        plt.title("NCSN DSM Loss")
        plt.legend()
        plt.tight_layout()
        plt.savefig(output_dir / "ncsn_loss.png")
        plt.close()
    except Exception:
        # Best-effort plotting
        pass

    return history


# Example: full_ncsn_cfg = NCSNConfig(epochs=30)
# full_train_ncsn(full_ncsn_cfg, paths.images / "ncsn", conditional=False)
# full_train_ncsn(full_ncsn_cfg, paths.images / "ncsn_cond", conditional=True)

### NCSN Conditional: Generate a 10x10 class-conditional grid

Create a grid where each row corresponds to a digit label 0–9, and each row contains multiple generated samples for that class. The results are saved to `images/ncsn_cond_grid.png`.

In [ ]:
# Heavy generation cell: run full pipelines to produce report images (compute-heavy)
# Set RUN_FULL=True to execute; this cell includes OOM handling and CPU fallback.
RUN_FULL = False  # set to True to run full generation (use GPU runtime if available)
FORCE = False     # set True to retrain even if checkpoints exist
CPU_FALLBACK = True  # if GPU OOM occurs, retry on CPU with reduced settings

if RUN_FULL:
    print('Starting full report-generation run...')
    from pathlib import Path
    import torch
    from config import DataConfig, EBMConfig, NCSNConfig, RunPaths
    from ebm_train import train as full_train_ebm
    from ncsn_train import train as full_train_ncsn
    from ebm_infer import generate_and_denoise as ebm_generate_and_denoise, sample_and_save_trajectory as ebm_sample_traj
    from ncsn_infer import ncsn_generate_and_denoise, load_ncsn_model
    from ncsn_sampling import sample as ncsn_sample
    from utils import save_grid

    paths = RunPaths()
    paths.ensure()

    def try_train_ebm(cfg, out_dir):
        try:
            full_train_ebm(DataConfig(), cfg, out_dir)
        except RuntimeError as e:
            if 'out of memory' in str(e).lower() and CPU_FALLBACK:
                print('EBM OOM on GPU; retrying on CPU with reduced steps...')
                try:
                    torch.cuda.empty_cache()
                except Exception:
                    pass
                cfg.device = torch.device('cpu')
                cfg.langevin_steps = max(8, cfg.langevin_steps // 4)
                full_train_ebm(DataConfig(), cfg, out_dir)
            else:
                raise

    def try_train_ncsn(cfg, out_dir, conditional=False):
        try:
            full_train_ncsn(cfg, out_dir, conditional=conditional)
        except RuntimeError as e:
            msg = str(e).lower()
            if 'out of memory' in msg and CPU_FALLBACK:
                print('NCSN OOM detected; attempting staged fallback...')
                try:
                    torch.cuda.empty_cache()
                except Exception:
                    pass

                # Stage 1: Try smaller batch on GPU
                old_batch = cfg.batch_size
                cfg.batch_size = max(8, cfg.batch_size // 4)
                print(f'Retrying on GPU with reduced batch_size={cfg.batch_size} (was {old_batch})...')
                try:
                    full_train_ncsn(cfg, out_dir, conditional=conditional)
                    return
                except RuntimeError as e2:
                    msg2 = str(e2).lower()
                    if 'out of memory' in msg2 and CPU_FALLBACK:
                        print('Still OOM on GPU; switching to CPU with aggressive reductions...')
                        # Switch device to CPU and apply aggressive reductions
                        cfg.device = torch.device('cpu')
                        cfg.num_levels = max(3, cfg.num_levels // 4)
                        cfg.langevin_steps = max(5, cfg.langevin_steps // 10)
                        cfg.batch_size = min(cfg.batch_size, 16)
                        # Reduce model size where applicable
                        if hasattr(cfg, 'embed_dim') and cfg.embed_dim > 128:
                            print(f'Reducing embed_dim from {cfg.embed_dim} to 128 for CPU run')
                            cfg.embed_dim = 128
                        # Optionally reduce other heavy parameters (user can customize further)
                        full_train_ncsn(cfg, out_dir, conditional=conditional)
                        return
                    else:
                        raise
            else:
                raise

    # EBM full run
    ebm_out = paths.images / 'ebm'
    ebm_cfg = EBMConfig()
    ebm_cfg.epochs = 10
    ebm_cfg.sample_grid = 16
    ebm_cfg.langevin_steps = 60
    if FORCE or not (ebm_out / 'ebm_ckpt.pt').exists():
        try_train_ebm(ebm_cfg, ebm_out)
    else:
        print('EBM checkpoint exists; skipping training')

    # EBM inference
    try:
        ck = ebm_out / 'ebm_ckpt.pt'
        if ck.exists():
            ebm_generate_and_denoise(ck, paths.images / 'ebm_infer', DataConfig(), ebm_cfg)
            ebm_sample_traj(ck, paths.images / 'ebm_traj_demo', ebm_cfg, record_every=5)
        else:
            print('EBM checkpoint missing; inference skipped')
    except Exception as e:
        print('EBM inference failed:', e)

    # NCSN full run (unconditional + conditional)
    ncsn_out = paths.images / 'ncsn'
    ncsn_cond_out = paths.images / 'ncsn_cond'
    ncsn_cfg = NCSNConfig()
    ncsn_cfg.epochs = 30
    ncsn_cfg.num_levels = 10
    ncsn_cfg.langevin_steps = 150

    if FORCE or not (ncsn_out / 'ncsn.pt').exists():
        try_train_ncsn(ncsn_cfg, ncsn_out, conditional=False)
    else:
        print('NCSN checkpoint exists; skipping unconditional training')

    if FORCE or not (ncsn_cond_out / 'ncsn_cond.pt').exists():
        try_train_ncsn(ncsn_cfg, ncsn_cond_out, conditional=True)
    else:
        print('NCSN checkpoint exists; skipping conditional training')

    # NCSN inference
    try:
        ck1 = ncsn_out / 'ncsn.pt'
        if ck1.exists():
            ncsn_generate_and_denoise(ck1, paths.images / 'ncsn_infer', ncsn_cfg, conditional=False)
        else:
            print('NCSN unconditional checkpoint missing; inference skipped')
    except Exception as e:
        print('NCSN inference failed:', e)

    try:
        ck2 = ncsn_cond_out / 'ncsn_cond.pt'
        if ck2.exists():
            ncsn_generate_and_denoise(ck2, paths.images / 'ncsn_cond_infer', ncsn_cfg, conditional=True)
        else:
            print('NCSN conditional checkpoint missing; inference skipped')
    except Exception as e:
        print('NCSN conditional inference failed:', e)

    # NCSN trajectory generation
    try:
        if (ncsn_out / 'ncsn.pt').exists():
            model_n = load_ncsn_model(ncsn_out / 'ncsn.pt', ncsn_cfg, conditional=False)
            traj = ncsn_sample(model_n, ncsn_cfg, num_samples=16, return_trajectory=True, record_every=10)
            out_dir_n = paths.images / 'ncsn_traj_demo'
            out_dir_n.mkdir(parents=True, exist_ok=True)
            for i, frame in enumerate(traj):
                save_grid((frame + 1) / 2.0, out_dir_n / f'ncsn_traj_{i:03d}.png', nrow=4)
        else:
            print('Skipping NCSN trajectory generation; checkpoint missing')
    except Exception as e:
        print('NCSN trajectory generation failed:', e)

    print('Full report image generation complete. Look under the images/ directory.')
else:
    print('Heavy generation is disabled. Set RUN_FULL = True to enable.')

# NCSN Conditional: Generate a 10x10 class-conditional grid

In [ ]:
# Conditional generation grid
import torch
from ncsn_infer import load_ncsn_model
from ncsn_sampling import sample
from utils import save_grid

ncsn_cond_ckpt = paths.images / "ncsn_cond" / "ncsn_cond.pt"
if ncsn_cond_ckpt.exists():
    cfg = NCSNConfig(conditional=True)
    model = load_ncsn_model(ncsn_cond_ckpt, cfg, conditional=True)
    grid_dir = paths.images / "ncsn_cond_grid"
    grid_dir.mkdir(exist_ok=True)
    all_samples = []
    for digit in range(10):
        y = torch.full((10,), digit, device=cfg.device, dtype=torch.long)
        samples = sample(model, cfg, num_samples=10, y=y)
        all_samples.append(samples)
    # Concatenate into 10x10 grid
    grid = torch.cat([torch.cat(row, dim=0) for row in all_samples], dim=1)
    save_grid((grid + 1) / 2.0, grid_dir / "ncsn_cond_grid.png", nrow=10)
    print("Conditional grid saved to", grid_dir / "ncsn_cond_grid.png")
else:
    print("Conditional NCSN checkpoint not found.")

# NCSN: Annealed Langevin Dynamics Trajectory

In [ ]:
# NCSN trajectory
from ncsn_infer import load_ncsn_model
from ncsn_sampling import sample

ncsn_ckpt = paths.images / "ncsn" / "ncsn.pt"
if ncsn_ckpt.exists():
    cfg = NCSNConfig()
    model = load_ncsn_model(ncsn_ckpt, cfg, conditional=False)
    traj_dir = paths.images / "ncsn_trajectory"
    traj_dir.mkdir(exist_ok=True)
    trajectory = sample(model, cfg, num_samples=1, return_trajectory=True, record_every=10)
    for i, frame in enumerate(trajectory):
        save_grid((frame + 1) / 2.0, traj_dir / f"ncsn_traj_{i:03d}.png", nrow=1)
    print("Trajectory frames saved to", traj_dir)
    make_gif_from_frames(traj_dir, paths.images / "ncsn_trajectory.gif", fps=6)
else:
    print("NCSN checkpoint not found.")

## Usage Notes

- Full training pipelines are available in the notebook: `full_train_ebm` and `full_train_ncsn`.
- Inference pipelines: `ebm_generate_and_denoise` and `ncsn_generate_and_denoise`.
- For quick demos, use the short training functions with `epochs=1`.
- Ensure `torch` and `torchvision` are installed (see `requirements.txt`).
- Figures are saved to `paths.images` subdirectories; inline plots are for demos.

# Theory Questions: Score-Based Models (NCSN)

**Q2.1:** The score function is $s_\theta(x) = \nabla_x \log p_\theta(x) = -\nabla_x E_\theta(x)$. It is independent of the partition function $Z$ because $\nabla_x \log Z = 0$. This makes training easier since we do not need to compute $Z$.

**Q2.2:** Calculating $\nabla_x \cdot s_\theta(x)$ (divergence) is expensive in high dimensions. DSM replaces this with a denoising objective, which is equivalent to score matching on the noisy distribution, making it tractable.

**Q2.3:** In each region, the score depends only on the local density, not the mixture weights. Thus, Langevin dynamics cannot sample according to the mixture weights and cannot move between disjoint modes.

**Q2.4:** Direct score matching on real data fails due to low support and non-smoothness. Multi-scale noise perturbation (as in NCSN) smooths the data and enables learning. $\sigma_{max}$ controls the largest noise (most smoothing), $\sigma_{min}$ the finest details in annealed Langevin dynamics.

# Detailed Theory: Score-Based Models (NCSN)

## Q2.1 — Score function and independence from the partition function

Definition: the score function is s(x) = ∇_x log p(x). For an energy parameterization p(x) ∝ exp(-E(x)),

log p(x) = -E(x) - log Z, so ∇_x log p(x) = -∇_x E(x) - ∇_x log Z = -∇_x E(x).

Because Z does not depend on x, ∇_x log Z = 0. Therefore the score depends only on E (or the unnormalized density), not on Z. Advantage: learning the score avoids computing or approximating Z (the partition function), which is usually intractable for high-dimensional models.

## Q2.2 — Why the divergence term is hard, and Denoising Score Matching (DSM)

Original score-matching objectives involve the trace (divergence) term ∇_x·s(x) = ∑_i ∂ s_i / ∂ x_i which requires computing many partial derivatives (second derivatives of the underlying model) and scales poorly with dimensionality (images have very large D).

Denoising Score Matching (DSM) idea (intuitive): instead of directly minimizing an objective containing divergence, we corrupt data with Gaussian noise and minimize the expected squared error between the model score and the true score of the noisy distribution: E_{x, x̃∼N(x,σ^2I)} || s_θ(x̃) − ∇_{x̃} log p_σ(x̃ | x) ||^2. This avoids the divergence term and is tractable because the score of the noisy conditional is known analytically for Gaussian noise. In practice this gives an efficient surrogate for the original score-matching objective.

## Q2.3 — Mixture of Gaussians with disjoint supports (effect of mixing weights)

Consider p(x) = π_A p_A(x) + π_B p_B(x). If supports are disjoint (region A where p_B=0 and region B where p_A=0), then for x ∈ A:

log p(x) = log(π_A p_A(x)) = log π_A + log p_A(x),
so ∇ log p(x) = ∇ log p_A(x) (the constant log π_A disappears).

Therefore the score in region A depends only on the local component density p_A, not the mixing weight π_A. Practical consequence: Langevin dynamics uses local gradients and cannot account for global mixing weights, so it will not sample the relative frequencies of the mixture correctly and modes remain isolated (mixing problem). The sampler rarely jumps between disjoint modes because the gradient signal points inward toward the mode and the energy barrier between modes is high.

## Q2.4 — Problems with score matching on raw data and how NCSN fixes them

Problems with direct score matching on real data:
- Real data often lie on a low-dimensional manifold embedded in high-dimensional space; the density off the manifold is effectively zero and the score can be ill-defined or numerically unstable.
- The empirical distribution is spiky; learning the score at exact data points does not generalize well to neighborhoods.

NCSN / Multi-scale noise solution:
- Perturb the data with multiple noise levels (a geometric sequence of σ values). This smooths the data distribution at different scales and makes the score well-defined everywhere.
- Train a model to predict scores conditioned on the noise level (Noise Conditional Score Network).
- Annealed Langevin Dynamics (ALD) sampling: start from large noise σ_max (smooth, easy to explore) and gradually reduce noise to σ_min, performing Langevin updates at each noise level. σ_max encourages global exploration and helps cross modes; σ_min recovers fine details and makes samples sharp.

## References and further reading
- Hyv arinen, "Estimation of non-normalized statistical models by score matching" (2005).
- Song & Ermon, "Generative Modeling by Estimating Gradients of the Data Distribution" (NCSN, 2019).
- Vincent, "A Connection Between Score Matching and Denoising Autoencoders" (2011).

# Complete Theory (Score-Based Models / NCSN) — Detailed Derivations, Algorithms, and Practical Tips

## 1. Score function and independence from normalization (Q2.1 expanded)

For p(x) ∝ exp(-E(x)), the score is s(x) = ∇_x log p(x) = −∇_x E(x). Because the normalization constant Z is independent of x, ∇_x log Z = 0, so the score contains no dependence on Z. This is a key advantage: we can learn the gradient field of log-density without ever estimating Z.

## 2. Why divergence is hard and DSM equivalence (Q2.2 expanded)

Original score matching objectives include the divergence term ∇_x · s(x) which requires computing second derivatives of the model w.r.t. inputs (trace of Jacobian) and scales poorly with dimensionality (O(D^2) naively).

Denoising Score Matching (DSM) circumvents this by training a model to predict the score of the noisy distribution p_σ(x̃) = ∫ p(x) N(x̃ | x, σ^2 I) dx, using the simple closed form score for Gaussian corruption. The DSM loss for noise level σ is: L_σ(θ) = E_{x ∼ p_data, x̃ ∼ N(x,σ^2 I)} [ || s_θ(x̃, σ) − ∇_{x̃} log N(x̃ | x, σ^2 I) ||^2 ]
Minimizing this loss is equivalent to matching the score of the smoothed density p_σ, which regularizes learning and avoids computing divergences.

## 3. Mixing/modes and score-based failure modes (Q2.3 expanded)

If the data distribution is a mixture of well-separated components, the local score in each component is determined only by that component's density; mixing weights cancel out (log π_k term disappears when differentiating wrt x). Consequently, local gradient-based samplers (Langevin) tend to remain trapped in modes and fail to sample according to mixture weights (rare-mode under-sampling). Annealed (multi-scale) approaches help but do not fully solve extreme imbalance without other techniques (tempering, MCMC with global proposals, or importance sampling).

## 4. NCSN and Multi-scale DSM (Q2.4 expanded)

NCSN trains a noise-conditional score network s_θ(x, σ) to predict the score under different noise levels σ ∈ {σ_1, …, σ_L}. The common training objective is a weighted sum:

L(θ) = Σ_{i=1}^L λ(σ_i) E_{x, x̃ ∼ N(x, σ_i^2 I)} [ || s_θ(x̃, σ_i) − (−(x̃ − x)/σ_i^2 ) ||^2 ],

where λ(σ) is a weighting usually chosen to balance scales (often λ(σ) ∝ σ^2 or inverse variance). At inference time, Annealed Langevin Dynamics (ALD) runs Langevin sampling starting from noise at σ_max and anneals σ down to σ_min, performing several Langevin steps at each σ.

ALD pseudocode:
```python
x = Normal(0, σ_max^2)
for σ in schedule(σ_max->σ_min):
    for t in range(T_σ):
        grad = s_θ(x, σ)
        x = x + α_σ * grad + sqrt(2 α_σ) * Normal(0, I)
```

Practical choices:
- σ_max should be large enough to smooth data and enable exploration; σ_min small enough for fine details.
- T_σ (steps per level) and α_σ (step sizes) must be tuned; more steps give better quality at greater cost.

## 5. Conditional NCSN (brief notes)

- Add class embedding e_y (nn.Embedding) and combine with noise embedding (e.g., Gaussian Fourier features) before injecting into AdaptiveResBlocks using FiLM (scale & shift).
- Train with labeled data by conditioning s_θ(x̃, σ, y) and compute DSM loss using the label-conditioned scores.
- At sampling time provide desired labels y and run ALD conditioned on y to obtain class-specific samples.

## 6. Architectures, hyperparameters and tips

Suggested hyperparameters (starting point):
- σ_max = 30, σ_min = 0.01, L = 10 levels (geometric), batch size = 64, LR = 2e-4, epochs = 30–100, T_σ = 50–150 steps per level in final runs (less for demos).

Architecture notes:
- Use U-Net style ScoreNet with AdaptiveResBlocks and FiLM conditioning for σ and y.
- Gaussian Fourier embeddings (GFP) for scalar σ help encode scale information into the network.

Diagnostics:
- Track validation denoising error per σ, visualize intermediate ALD frames (trajectory GIFs), and inspect class-conditional grids for the conditional model.
- If samples are noisy, increase T_σ or tune α_σ; if training diverges, reduce LR or increase gradient clipping.

## 7. Short references (use these to cite in your report)
- Hyv arinen, "Estimation of non-normalized statistical models by score matching" (2005).
- Vincent, "A Connection Between Score Matching and Denoising Autoencoders" (2011).
- Song & Ermon, "Generative Modeling by Estimating Gradients of the Data Distribution" (NCSN, 2019).
- Song et al., "Score-Based Generative Modeling through Stochastic Differential Equations" (2021).

In [ ]:
## Visualization helpers (save and display rich visualizations)
import imageio
import matplotlib.pyplot as plt
from pathlib import Path


def show_image(path: Path, figsize=(5, 5)):
    img = plt.imread(path)
    plt.figure(figsize=figsize)
    plt.axis('off')
    plt.imshow(img)
    plt.show()


def make_gif_from_frames(frames_dir: Path, out_path: Path, fps: int = 6):
    frames = sorted(frames_dir.glob("*.png"))
    if not frames:
        print(f"No frames found in {frames_dir}")
        return
    imgs = [imageio.imread(str(p)) for p in frames]
    imageio.mimsave(str(out_path), imgs, fps=fps)
    print(f"Saved GIF: {out_path}")


def display_saved_run_images(run_images_dir: Path):
    """Display all images saved in a run directory in sorted order."""
    files = sorted(run_images_dir.glob("*.png"))
    for f in files:
        show_image(f)


# Note: imageio is optional; if not available, the GIF creation will raise an ImportError.

In [ ]:
# EBM Inference Pipeline: Generation and Denoising
from pathlib import Path
import torch

from config import DataConfig, EBMConfig, RunPaths
from data import mnist_dataloaders
from ebm_model import ConvEnergyModel
from ebm_sampling import sample_from_noise, LangevinSampler
from utils import save_grid, set_seed, ensure_dir


def load_ebm_model(checkpoint: Path, device: torch.device) -> ConvEnergyModel:
    model = ConvEnergyModel().to(device)
    state = torch.load(checkpoint, map_location=device)
    model.load_state_dict(state["model"])
    model.eval()
    return model


def ebm_generate_and_denoise(
    checkpoint: Path, output_dir: Path, data_cfg: DataConfig, ebm_cfg: EBMConfig
) -> None:
    set_seed(data_cfg.seed)
    train_loader, _ = mnist_dataloaders(data_cfg)
    ensure_dir(output_dir)
    model = load_ebm_model(checkpoint, ebm_cfg.device)
    sampler = LangevinSampler(model, ebm_cfg)

    # Sampling requires gradients (Langevin uses autograd on inputs); do not disable grads here.
    samples = sample_from_noise(model, ebm_cfg, (16, 1, 28, 28))
    save_grid(samples.detach().cpu(), output_dir / "ebm_samples_final.png", nrow=4)

    # Denoise a few training digits
    x_real, _ = next(iter(train_loader))
    x_real = x_real[:16].to(ebm_cfg.device)
    noise = torch.randn_like(x_real) * 0.3
    noisy = (x_real + noise).clamp(0.0, 1.0)
    denoised = sampler(noisy)
    save_grid(x_real.detach().cpu(), output_dir / "ebm_real.png", nrow=4)
    save_grid(noisy.detach().cpu(), output_dir / "ebm_noisy.png", nrow=4)
    save_grid(denoised.detach().cpu(), output_dir / "ebm_denoised.png", nrow=4)


# Example: ebm_generate_and_denoise(paths.images / "ebm" / "ebm_ckpt.pt", paths.images / "ebm_infer", DataConfig(), EBMConfig())

In [ ]:
# Quick import test for EBM inference helpers (no running of heavy functions)
try:
    from ebm_infer import generate_and_denoise, sample_and_save_trajectory
    print('Imported EBM inference helpers successfully')
except Exception as e:
    print('Failed to import EBM inference helpers:', e)

# Quick import test for NCSN inference helpers
try:
    from ncsn_infer import ncsn_generate_and_denoise, load_ncsn_model
    print('Imported NCSN inference helpers successfully')
except Exception as e:
    print('Failed to import NCSN inference helpers:', e)

In [ ]:
# NCSN Inference Pipeline: Sampling and Denoising
from pathlib import Path
from typing import Optional, Sequence
import torch

from config import NCSNConfig, DataConfig, RunPaths
from data import mnist_dataloaders
from ncsn_model import ScoreNet
from ncsn_sampling import sample, annealed_langevin_dynamics
from utils import save_grid, ensure_dir


def load_ncsn_model(
    checkpoint: Path, cfg: NCSNConfig, conditional: bool = False
) -> ScoreNet:
    cfg.conditional = conditional
    model = ScoreNet(cfg).to(cfg.device)
    state = torch.load(checkpoint, map_location=cfg.device)
    model.load_state_dict(state["model"])
    model.eval()
    return model


@torch.no_grad()
def ncsn_generate_and_denoise(
    checkpoint: Path,
    output_dir: Path,
    cfg: NCSNConfig,
    conditional: bool = False,
    noise_levels: Sequence[float] = (0.2, 0.4, 0.6),
) -> None:
    ensure_dir(output_dir)
    data_cfg = DataConfig(batch_size=16)
    train_loader, _ = mnist_dataloaders(data_cfg, normalize_to_minus1_1=True)
    model = load_ncsn_model(checkpoint, cfg, conditional)

    y_samples: Optional[torch.Tensor] = None
    if conditional:
        y_samples = torch.arange(0, 16, device=cfg.device) % cfg.num_classes
    samples = sample(model, cfg, num_samples=16, y=y_samples)
    save_grid((samples + 1) / 2.0, output_dir / "ncsn_samples.png", nrow=4)

    x_real, labels = next(iter(train_loader))
    x_real = x_real.to(cfg.device)[:16] * 2 - 1
    y = labels.to(cfg.device)[:16] if conditional else None

    for nl in noise_levels:
        noisy = x_real + nl * torch.randn_like(x_real)
        sigmas = torch.tensor([nl], device=cfg.device)
        denoised = annealed_langevin_dynamics(model, cfg, sigmas, noisy.clone(), y)
        save_grid((noisy + 1) / 2.0, output_dir / f"noisy_{nl:.2f}.png", nrow=4)
        save_grid((denoised + 1) / 2.0, output_dir / f"denoised_{nl:.2f}.png", nrow=4)


# Example: ncsn_generate_and_denoise(paths.images / "ncsn" / "ncsn.pt", paths.images / "ncsn_infer", NCSNConfig(), conditional=False)

### EBM: Denoising at multiple noise levels (0.2, 0.4, 0.6)

Run the following cell to generate EBM samples and denoise 16 training images using the specified noise levels. The outputs will be saved to the images directory and include `ebm_real_<nl>.png`, `ebm_noisy_<nl>.png`, and `ebm_denoised_<nl>.png` for each noise level.

In [ ]:
# Run multi-noise denoising using EBM inference helper (no execution here)
from ebm_infer import generate_and_denoise as ebm_generate_and_denoise

ebm_ck = paths.images / 'ebm' / 'ebm_ckpt.pt'
out = paths.images / 'ebm_infer_noise_levels'
if ebm_ck.exists():
    ebm_generate_and_denoise(ebm_ck, out, DataConfig(), EBMConfig(), noise_levels=[0.2, 0.4, 0.6])
else:
    print('EBM checkpoint missing; train EBM or place checkpoint at', ebm_ck)


### EBM: Post-training sampling from training images (use images as initialization)

This cell uses a trained EBM model to perform Langevin sampling starting from a small batch of training images (not noise). This visualizes how the model refines/changes real images under its energy field and is useful for analysis in the report.

In [ ]:
# Use training images as starting points for Langevin sampling and save results
from ebm_infer import load_model
from ebm_sampling import LangevinSampler

ck = paths.images / 'ebm' / 'ebm_ckpt.pt'
out = paths.images / 'ebm_posttrain_init'
out.mkdir(parents=True, exist_ok=True)

if ck.exists():
    cfg = EBMConfig()
    model = load_model(ck, cfg.device)
    sampler = LangevinSampler(model, cfg)
    train_loader, _ = mnist_dataloaders(DataConfig(), normalize_to_minus1_1=False)
    x_real, _ = next(iter(train_loader))
    x_real = x_real[:16].to(cfg.device)
    # Run sampling starting from the images directly
    x_sampled = sampler(x_real)
    save_grid(x_real.detach().cpu(), out / 'post_init_real.png', nrow=4)
    save_grid(x_sampled.detach().cpu(), out / 'post_init_sampled.png', nrow=4)
    print('Saved post-training init images to', out)
else:
    print('No EBM checkpoint found at', ck)

## Quick demo run (short, inline)

- Runs 1 epoch EBM and NCSN (unconditional) with default configs.
- Uses small epochs to keep runtime manageable; for full quality use the scripts.
- Displays sample grids inline (not saved); ensure `torch` is installed and GPU is recommended.


# Analysis: EBM Results

- The EBM model is trained using contrastive divergence and Langevin sampling.
- Generated samples and denoised images are saved after each epoch.
- The quality of generated images improves with training, but may be blurry or lack diversity if the model or sampling steps are insufficient.
- Denoising works well for moderate noise, but fails for very high noise levels.
- See saved images in the results directory for qualitative evaluation.

In [ ]:
# Langevin walkthrough demo (small, quick to run)
from pathlib import Path
import torch
from torchvision.utils import make_grid, save_image
import matplotlib.pyplot as plt
from IPython.display import Image, display

from utils import set_seed, ensure_dir
from ebm_model import ConvEnergyModel
from data import mnist_dataloaders

set_seed(42)

out_dir = paths.images / 'ebm_langevin_walkthrough'
ensure_dir(out_dir)

# Load or initialize model
ck = paths.images / 'ebm' / 'ebm_ckpt.pt'
model = ConvEnergyModel().to(device)
if ck.exists():
    try:
        state = torch.load(ck, map_location=device)
        model.load_state_dict(state['model'])
        model.eval()
        print('Loaded EBM checkpoint for demo.')
    except Exception as e:
        print('Failed to load checkpoint, using random init:', e)
else:
    print('No checkpoint found; using random initialized model for demonstration.')

# Get one MNIST image (train) and prepare initial noise
train_loader, _ = mnist_dataloaders(data_cfg, normalize_to_minus1_1=False)
x_real, _ = next(iter(train_loader))
x_real = x_real[:1].to(device)
x_init = torch.rand_like(x_real).to(device)  # uniform [0,1]

# Langevin hyperparams (small demo)
T = 100
record_every = 10
eta = 0.1

x = x_init.clone()
energy_trace = []

frames_dir = out_dir / 'frames'
frames_dir.mkdir(parents=True, exist_ok=True)

for t in range(1, T + 1):
    x.requires_grad_(True)
    E = model(x).mean()
    grad = torch.autograd.grad(E, x)[0]
    with torch.no_grad():
        x = x - 0.5 * eta * grad + torch.randn_like(x) * (eta ** 0.5)
        x = x.clamp(0.0, 1.0)
    energy_trace.append(E.item())
    if t % record_every == 0 or t == 1:
        save_image(make_grid(x.detach().cpu(), nrow=1), frames_dir / f'frame_{t:03d}.png')

# Save final image and create a grid with initial / final / real
save_image(make_grid(x_init.detach().cpu(), nrow=1), out_dir / 'init.png')
save_image(make_grid(x.detach().cpu(), nrow=1), out_dir / 'final.png')
save_image(make_grid(x_real.detach().cpu(), nrow=1), out_dir / 'real.png')

# Plot energy over time
plt.figure(figsize=(6, 3))
plt.plot(energy_trace, label='Mean Energy')
plt.xlabel('Step')
plt.ylabel('Energy')
plt.title('EBM Mean Energy during Langevin Walkthrough')
plt.legend()
plt.tight_layout()
plt.savefig(out_dir / 'energy_trace.png')
plt.close()

# Create GIF (best-effort; make_gif_from_frames exists earlier in the notebook)
try:
    make_gif_from_frames(frames_dir, out_dir / 'langevin_walk.gif', fps=6)
    display(Image(str(out_dir / 'langevin_walk.gif')))
except Exception as e:
    print('Could not create GIF:', e)

# Show the saved images inline (init, final, real, energy)
display(Image(str(out_dir / 'init.png')))
display(Image(str(out_dir / 'final.png')))
display(Image(str(out_dir / 'real.png')))
display(Image(str(out_dir / 'energy_trace.png')))

# Parameter Sweep: Langevin Step Size (η) and Number of Steps (T)

This section runs a small sweep over combinations of step-size η and number of Langevin steps T to compare their qualitative effects on generated samples and the evolution of the model energy. It will:

- Run short Langevin chains starting from the same random initialization for fair comparison.
- Save final images in a single comparison grid where columns correspond to η and rows correspond to T.
- Save per-run energy traces to help diagnose dynamics.

Notes:
- For speed this sweep uses conservative defaults (quick mode). To run a fuller sweep set SWEEP_QUICK=False and increase the lists for `etas` and `Ts`.
- The outputs are saved to `images/ebm_langevin_sweep/` (create the directory if missing).

# Observations and Tips from the Walkthrough

- If a trained checkpoint is available, you will usually see meaningful digit-like structure emerge in the trajectory and the final sample; with a randomly initialized model the dynamics are noisy and do not produce digits.
- The energy plot helps diagnose whether the sampler is decreasing/increasing expected quantities; with correct dynamics, energy should trend downward for a trained EBM on data-like samples.
- If the trajectory looks unstable (large jumps), reduce η or add more noise smoothing; if samples do not change, increase η or T.
- For higher-quality generation increase T and use a trained model; this demo is intended as an educational visualization, not as a full-quality sampling run.

In [ ]:
# Parameter sweep: vary eta and T and save comparison grids
from pathlib import Path
import torch
import matplotlib.pyplot as plt
from torchvision.utils import make_grid, save_image

SWEEP_QUICK = True  # set False for a larger/more expensive sweep
if SWEEP_QUICK:
    etas = [0.01, 0.05, 0.1]
    Ts = [20, 100, 300]
else:
    etas = [0.005, 0.01, 0.02, 0.05, 0.1]
    Ts = [20, 50, 100, 300, 600]

out_root = paths.images / 'ebm_langevin_sweep'
out_root.mkdir(parents=True, exist_ok=True)
frames_root = out_root / 'frames'
frames_root.mkdir(parents=True, exist_ok=True)

# Load or initialize model (as in the walkthrough)
ck = paths.images / 'ebm' / 'ebm_ckpt.pt'
model = ConvEnergyModel().to(device)
if ck.exists():
    try:
        state = torch.load(ck, map_location=device)
        model.load_state_dict(state['model'])
        model.eval()
        print('Loaded EBM checkpoint for sweep.')
    except Exception as e:
        print('Failed to load checkpoint, using random init for sweep:', e)
else:
    print('No checkpoint found; using random initialized model for sweep. Results are illustrative only.')

# Single starting initialization for fair comparisons
set_seed(42)
train_loader, _ = mnist_dataloaders(data_cfg, normalize_to_minus1_1=False)
x_real, _ = next(iter(train_loader))
x_real = x_real[:1].to(device)
x_init = torch.rand_like(x_real).to(device)

# Containers
final_images = []  # will store final images row-major: for T in Ts: for eta in etas
summary_records = []

for i_T, T in enumerate(Ts):
    row_images = []
    for j_e, eta in enumerate(etas):
        x = x_init.clone()
        energy_trace = []
        for t in range(1, T + 1):
            x.requires_grad_(True)
            E = model(x).mean()
            grad = torch.autograd.grad(E, x)[0]
            with torch.no_grad():
                x = x - 0.5 * eta * grad + torch.randn_like(x) * (eta ** 0.5)
                x = x.clamp(0.0, 1.0)
            energy_trace.append(E.item())
        # Save per-run final image and energy trace
        run_name = f'eta{eta:.4f}_T{T}'
        save_image(make_grid(x.detach().cpu(), nrow=1), out_root / f'final_{run_name}.png')
        # Energy plot
        plt.figure(figsize=(4, 2))
        plt.plot(energy_trace)
        plt.xlabel('Step')
        plt.ylabel('Mean Energy')
        plt.title(run_name)
        plt.tight_layout()
        plt.savefig(out_root / f'energy_{run_name}.png')
        plt.close()

        row_images.append(x.detach().cpu())
        summary_records.append({'T': T, 'eta': eta, 'final_image': str(out_root / f'final_{run_name}.png'), 'energy_plot': str(out_root / f'energy_{run_name}.png')})
    # after finishing all etas for this T, create a row grid and save
    row_tensor = torch.cat(row_images, dim=0)
    save_image(make_grid(row_tensor, nrow=len(etas)), out_root / f'row_T{T}.png')
    final_images.append(row_tensor)

# Create the full grid (rows correspond to Ts)
all_rows = torch.cat(final_images, dim=0)
save_image(make_grid(all_rows, nrow=len(etas)), out_root / 'sweep_grid.png')
print('Saved sweep grid at', out_root / 'sweep_grid.png')

# Save a small CSV-style summary
import json
with open(out_root / 'sweep_summary.json', 'w') as f:
    json.dump(summary_records, f, indent=2)

# Display final grid inline if possible
try:
    display(Image(str(out_root / 'sweep_grid.png')))
except Exception:
    print('Sweep complete. Check the images at', out_root)

# Worked Example: Langevin Update Walkthrough (single MNIST image)

This worked example demonstrates how a single MNIST image evolves under Langevin dynamics for an EBM. It includes:

- A short description of the update rule and pseudocode.
- A runnable code cell that performs a small number of Langevin steps starting from uniform noise and saves intermediate frames.
- A plot of energy over time and a generated GIF of the trajectory.

Langevin update (discrete form):

x_{t+1} = x_t − 0.5 * η * ∇_x E_θ(x_t) + sqrt(η) * ξ_t,  where ξ_t ∼ N(0, I)

Notes:
- We take gradients w.r.t. the input using autograd (x.requires_grad_(True)).
- After each update we clamp pixels to a valid range (e.g., [0,1]).
- For demonstrations, small η and a modest number of steps (T=100) are sufficient to see how the sample moves; for high-quality sampling larger T is needed.

In [ ]:
# Heavy generation cell: run full pipelines to produce report images (compute-heavy)
# Set RUN_FULL=True to execute; this cell includes OOM handling and CPU fallback.
RUN_FULL = False  # set to True to run full generation (use GPU runtime if available)
FORCE = False     # set True to retrain even if checkpoints exist
CPU_FALLBACK = True  # if GPU OOM occurs, retry on CPU with reduced settings

if RUN_FULL:
    print('Starting full report-generation run...')
    from pathlib import Path
    import torch
    from config import DataConfig, EBMConfig, NCSNConfig, RunPaths
    from ebm_train import train as full_train_ebm
    from ncsn_train import train as full_train_ncsn
    from ebm_infer import generate_and_denoise as ebm_generate_and_denoise, sample_and_save_trajectory as ebm_sample_traj
    from ncsn_infer import ncsn_generate_and_denoise, load_ncsn_model
    from ncsn_sampling import sample as ncsn_sample
    from utils import save_grid

    paths = RunPaths()
    paths.ensure()

    def try_train_ebm(cfg, out_dir):
        try:
            full_train_ebm(DataConfig(), cfg, out_dir)
        except RuntimeError as e:
            if 'out of memory' in str(e).lower() and CPU_FALLBACK:
                print('EBM OOM on GPU; retrying on CPU with reduced steps...')
                try:
                    torch.cuda.empty_cache()
                except Exception:
                    pass
                cfg.device = torch.device('cpu')
                cfg.langevin_steps = max(8, cfg.langevin_steps // 4)
                full_train_ebm(DataConfig(), cfg, out_dir)
            else:
                raise

    def try_train_ncsn(cfg, out_dir, conditional=False):
        try:
            full_train_ncsn(cfg, out_dir, conditional=conditional)
        except RuntimeError as e:
            if 'out of memory' in str(e).lower() and CPU_FALLBACK:
                print('NCSN OOM on GPU; retrying on CPU with reduced config...')
                try:
                    torch.cuda.empty_cache()
                except Exception:
                    pass
                cfg.device = torch.device('cpu')
                cfg.num_levels = max(3, cfg.num_levels // 2)
                cfg.langevin_steps = max(10, cfg.langevin_steps // 5)
                cfg.batch_size = min(cfg.batch_size, 32)
                full_train_ncsn(cfg, out_dir, conditional=conditional)
            else:
                raise

    # EBM full run
    ebm_out = paths.images / 'ebm'
    ebm_cfg = EBMConfig()
    ebm_cfg.epochs = 10
    ebm_cfg.sample_grid = 16
    ebm_cfg.langevin_steps = 60
    if FORCE or not (ebm_out / 'ebm_ckpt.pt').exists():
        try_train_ebm(ebm_cfg, ebm_out)
    else:
        print('EBM checkpoint exists; skipping training')

    # EBM inference
    try:
        ck = ebm_out / 'ebm_ckpt.pt'
        if ck.exists():
            ebm_generate_and_denoise(ck, paths.images / 'ebm_infer', DataConfig(), ebm_cfg)
            ebm_sample_traj(ck, paths.images / 'ebm_traj_demo', ebm_cfg, record_every=5)
        else:
            print('EBM checkpoint missing; inference skipped')
    except Exception as e:
        print('EBM inference failed:', e)

    # NCSN full run (unconditional + conditional)
    ncsn_out = paths.images / 'ncsn'
    ncsn_cond_out = paths.images / 'ncsn_cond'
    ncsn_cfg = NCSNConfig()
    ncsn_cfg.epochs = 30
    ncsn_cfg.num_levels = 10
    ncsn_cfg.langevin_steps = 150

    if FORCE or not (ncsn_out / 'ncsn.pt').exists():
        try_train_ncsn(ncsn_cfg, ncsn_out, conditional=False)
    else:
        print('NCSN checkpoint exists; skipping unconditional training')

    if FORCE or not (ncsn_cond_out / 'ncsn_cond.pt').exists():
        try_train_ncsn(ncsn_cfg, ncsn_cond_out, conditional=True)
    else:
        print('NCSN checkpoint exists; skipping conditional training')

    # NCSN inference
    try:
        ck1 = ncsn_out / 'ncsn.pt'
        if ck1.exists():
            ncsn_generate_and_denoise(ck1, paths.images / 'ncsn_infer', ncsn_cfg, conditional=False)
        else:
            print('NCSN unconditional checkpoint missing; inference skipped')
    except Exception as e:
        print('NCSN inference failed:', e)

    try:
        ck2 = ncsn_cond_out / 'ncsn_cond.pt'
        if ck2.exists():
            ncsn_generate_and_denoise(ck2, paths.images / 'ncsn_cond_infer', ncsn_cfg, conditional=True)
        else:
            print('NCSN conditional checkpoint missing; inference skipped')
    except Exception as e:
        print('NCSN conditional inference failed:', e)

    # NCSN trajectory generation
    try:
        if (ncsn_out / 'ncsn.pt').exists():
            model_n = load_ncsn_model(ncsn_out / 'ncsn.pt', ncsn_cfg, conditional=False)
            traj = ncsn_sample(model_n, ncsn_cfg, num_samples=16, return_trajectory=True, record_every=10)
            out_dir_n = paths.images / 'ncsn_traj_demo'
            out_dir_n.mkdir(parents=True, exist_ok=True)
            for i, frame in enumerate(traj):
                save_grid((frame + 1) / 2.0, out_dir_n / f'ncsn_traj_{i:03d}.png', nrow=4)
        else:
            print('Skipping NCSN trajectory generation; checkpoint missing')
    except Exception as e:
        print('NCSN trajectory generation failed:', e)

    print('Full report image generation complete. Look under the images/ directory.')
else:
    print('Heavy generation is disabled. Set RUN_FULL = True to enable.')

## Visualize demo losses

Plot the loss histories from the short demo runs to quickly inspect optimization behavior.


# Analysis: NCSN Results

- The NCSN model (unconditional and conditional) is trained with weighted DSM loss and annealed Langevin dynamics.
- Generated samples are sharper and more diverse than EBM, especially with sufficient noise levels and steps.
- Conditional NCSN can generate specific digits by conditioning on class labels.
- Denoising is effective for moderate noise, but extreme noise still degrades results.
- See saved images and GIFs for qualitative evaluation.

In [ ]:
import matplotlib.pyplot as plt

if "ebm_hist" in locals() and ebm_hist:
    plt.figure(figsize=(6, 4))
    plt.plot(ebm_hist["loss"])
    plt.title("EBM Loss (demo)")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.savefig(paths.images / "ebm_demo_loss.png")
    plt.show()
else:
    print("Run the demo cell to populate ebm_hist.")

if "ncsn_hist" in locals() and ncsn_hist:
    plt.figure(figsize=(6, 4))
    plt.plot(ncsn_hist["loss"])
    plt.title("NCSN DSM Loss (demo)")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.savefig(paths.images / "ncsn_demo_loss.png")
    plt.show()
else:
    print("Run the demo cell to populate ncsn_hist.")

### NCSN: Visualize sample trajectories (noise → sample) for 3 examples

This cell will assemble trajectory frames saved by the sampling utilities into GIFs for quick visual inspection. If trajectories have not been generated yet, run the trajectory generation in the heavy generation cell or invoke `annealed_langevin_dynamics_with_trajectory` directly.

In [ ]:
# Gather report-ready images into a single folder for convenience (does not run unless executed)
from shutil import copyfile
report_dir = paths.images / 'report_images'
report_dir.mkdir(parents=True, exist_ok=True)

candidates = {
    'mnist_sample.png': paths.images / 'mnist_sample.png',
    'ebm_samples.png': paths.images / 'ebm' / 'ebm_samples_epoch10.png',
    'ebm_denoised_0.20.png': paths.images / 'ebm_infer' / 'ebm_denoised_0.20.png',
    'ebm_denoised_0.40.png': paths.images / 'ebm_infer' / 'ebm_denoised_0.40.png',
    'ebm_denoised_0.60.png': paths.images / 'ebm_infer' / 'ebm_denoised_0.60.png',
    'ncsn_samples.png': paths.images / 'ncsn' / 'samples_epoch30.png',
    'ncsn_cond_grid.png': paths.images / 'ncsn_cond_grid' / 'ncsn_cond_grid.png',
}

copied = []
for name, p in candidates.items():
    if p.exists():
        copyfile(str(p), str(report_dir / name))
        copied.append(name)

print('Copied files to report folder:', copied)
print('Report images directory:', report_dir)


In [ ]:
# Create GIFs for NCSN trajectories if the frames exist
from pathlib import Path
from IPython.display import Image, display

def make_gif(frames_dir: Path, out_path: Path, fps: int = 6):
    if not frames_dir.exists():
        print('No frames directory at', frames_dir); return None
    frames = sorted(frames_dir.glob('*.png'))
    if not frames:
        print('No frames found in', frames_dir); return None
    imageio.mimsave(str(out_path), [imageio.imread(str(p)) for p in frames], fps=fps)
    print('Saved GIF', out_path)
    return out_path

ncsn_traj_dir = paths.images / 'ncsn_traj_demo'
if ncsn_traj_dir.exists():
    # Attempt to group frames into three sample GIFs by index naming pattern
    # If frames are saved as ncsn_traj_###.png, we'll just create a single GIF
    out = ncsn_traj_dir / 'ncsn_traj.gif'
    try:
        make_gif(ncsn_traj_dir, out, fps=6)
        display(Image(str(out)))
    except Exception as e:
        print('Could not create NCSN trajectory GIF:', e)
else:
    print('No NCSN trajectory frames found at', ncsn_traj_dir)


In [ ]:
# Create GIFs from trajectory frames if available and display saved demo outputs
import imageio
from pathlib import Path
from IPython.display import Image, display


def make_gif_from_dir(frames_dir: Path, out_path: Path, fps: int = 6):
    frames = sorted(frames_dir.glob("*.png"))
    if not frames:
        print(f"No frames found in {frames_dir}")
        return None
    imgs = [imageio.imread(str(p)) for p in frames]
    out_path.parent.mkdir(parents=True, exist_ok=True)
    imageio.mimsave(str(out_path), imgs, fps=fps)
    print(f"Saved GIF: {out_path}")
    return out_path

# EBM demo outputs
ebm_demo = paths.images / "ebm_demo"
if ebm_demo.exists():
    print("EBM demo outputs:")
    for p in sorted(ebm_demo.glob("*.png")):
        display(Image(str(p)))
else:
    print("No EBM demo outputs found at:", ebm_demo)

# Create/show EBM trajectory GIF if possible
ebm_traj_dir = paths.images / "ebm_traj_demo"
if ebm_traj_dir.exists():
    gif = make_gif_from_dir(ebm_traj_dir, ebm_traj_dir / "ebm_traj.gif")
    if gif:
        display(Image(str(gif)))

# NCSN demo outputs
ncsn_demo = paths.images / "ncsn_demo"
if ncsn_demo.exists():
    print("NCSN demo outputs:")
    for p in sorted(ncsn_demo.glob("*.png")):
        display(Image(str(p)))
else:
    print("No NCSN demo outputs found at:", ncsn_demo)

# Create/show NCSN trajectory GIF if possible
ncsn_traj_dir = paths.images / "ncsn_traj_demo"
if ncsn_traj_dir.exists():
    gif = make_gif_from_dir(ncsn_traj_dir, ncsn_traj_dir / "ncsn_traj.gif")
    if gif:
        display(Image(str(gif)))


In [ ]:
## Saved demo outputs and GIFs

This cell collects saved images from the demo runs (`ebm_demo`, `ncsn_demo`) and any trajectory frames, creates GIFs when possible, and displays all results inline for easy inspection.


## Display saved figures if available

Checks for images produced by the script entrypoints (e.g., `images/ebm/ebm_samples_epoch10.png`) and shows them inline when present.


In [ ]:
from matplotlib import image as mpimg

candidates = [
    paths.images / "ebm" / "ebm_samples_epoch10.png",
    paths.images / "ebm" / "ebm_denoised_epoch10.png",
    paths.images / "ncsn" / "samples_epoch30.png",
    paths.images / "ncsn_cond" / "samples_epoch30.png",
    paths.images / "ncsn_infer" / "denoised_0.40.png",
]

for img_path in candidates:
    if img_path.exists():
        img = mpimg.imread(img_path)
        plt.figure(figsize=(5, 5))
        plt.axis("off")
        plt.title(img_path.name)
        plt.imshow(img)
        plt.show()
    else:
        print(f"Missing: {img_path}")

In [ ]:
# Quick verification: check that expected outputs exist (files saved by full runs)
from pathlib import Path
expected = [
    paths.images / 'mnist_sample.png',
    paths.images / 'ebm' / 'ebm_ckpt.pt',
    paths.images / 'ebm' / 'ebm_samples_epoch10.png',
    paths.images / 'ebm' / 'ebm_denoised_epoch10.png',
    paths.images / 'ncsn' / 'samples_epoch30.png',
    paths.images / 'ncsn_cond' / 'samples_epoch30.png',
    paths.images / 'ncsn_infer' / 'denoised_0.40.png',
    paths.images / 'ncsn_cond_infer' / 'denoised_0.40.png',
]

missing = [str(p) for p in expected if not p.exists()]
if missing:
    print('The following expected files are missing (they will appear after you run the full training/inference runs):')
    for m in missing:
        print('- ', m)
else:
    print('All expected files found.')
